# **Import Libraries**

In [108]:
# import the required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import coo_matrix
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.linalg import svds

In [109]:
# define a function to load the data
def load_data(fname:str) -> pd.DataFrame:
    """
    Loads the data from the given file name and returns a pandas DataFrame.
    
    Parameters:
    fname (str): The name of the file to load the data from.
    
    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    # read the data from the file
    df = pd.read_csv(fname)

    # print the original data shape
    print(f'Original data shape: {df.shape}')
    
    return df

In [110]:
df_ratings = load_data('../data/processed/rating_data.csv')
df_ratings.head()

Original data shape: (100836, 3)


,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


# **Preprocessing Data**

## 1. Train-Test Split

In [111]:
train, test = train_test_split(df_ratings, test_size=0.2, random_state=42)

In [112]:
# sanity check train data
print(f'Train data shape: {train.shape}')
train.head()

Train data shape: (80668, 3)


,userId,movieId,rating
80568,509,7347,3.0
50582,326,71462,4.0
8344,57,2115,3.0
99603,610,1127,4.0
71701,462,2409,2.0


In [113]:
# sanity check test data
print(f'Test data shape: {test.shape}')
test.head()

Test data shape: (20168, 3)


,userId,movieId,rating
67037,432,77866,4.5
42175,288,474,3.0
93850,599,4351,3.0
6187,42,2987,4.0
12229,75,1610,4.0


## 2. Build Sparse User-Item Matrix (CSR)

In [114]:
# encode index
user_ids = train['userId'].astype('category').cat.codes
movie_ids = train['movieId'].astype('category').cat.codes

user_map = dict(enumerate(train['userId'].astype('category').cat.categories))
movie_map = dict(enumerate(train['movieId'].astype('category').cat.categories))

In [115]:
# build CSR matrix
matrix_coo = coo_matrix(
    (train['rating'],
    (user_ids, movie_ids))
)

user_item_matrix = matrix_coo.tocsr()

## 3. Set the Ground Truth (Test Set)

In [116]:
test_grouped = test.groupby('userId')['movieId'].apply(list).to_dict()

# **Modeling**

## 1. Cosine Similarity (Baseline)

In [117]:
item_similarity = cosine_similarity(user_item_matrix.T)

def recommend_cosine(user_idx, k=10):
    user_vector = user_item_matrix[user_idx].toarray().ravel()
    scores = item_similarity.dot(user_vector)
    top_items = np.argsort(scores)[::-1][:k]
    
    return top_items

## 2. KNN

In [ ]:
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(user_item_matrix.T)

def recommend_knn(user_idx, k=10):
    user_ratings = user_item_matrix[user_idx].toarray().ravel()
    liked_items = np.where(user_ratings > 0)[0]

    scores = {}

    for item in liked_items:
        distances, indices = knn.kneighbors(
            user_item_matrix.T[item].reshape(1, -1),
            n_neighbors=k
        )

        for i, dist in zip(indices.flatten(), distances.flatten()):
            scores[i] = scores.get(i, 0) + (1 - dist)

    top_items = sorted(scores, key=scores.get, reverse=True)[:k]

    return top_items

## 3. SVD (Matrix Factorization)

In [119]:
u, sigma, vt = svds(user_item_matrix, k=50)
sigma = np.diag(sigma)

pred_matrix = u @ sigma @ vt

def recommend_svd(user_idx, k=10):
    scores = pred_matrix[user_idx]

    return np.argsort(scores)[::-1][:k]

# **Evaluation Metrics**

## 1. Precision@K

Precision@K = (intersection of relevant & recommended) / K


In [120]:
def precision_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / k

def evaluate_model(model_func, k=10):
    precisions = []

    for user_id in test_grouped:
        if user_id not in user_map.values():
            continue
            
        user_idx = list(user_map.values()).index(user_id)
        recommended = model_func(user_idx, k)
        relevant = test_grouped[user_id]

        precisions.append(precision_at_k(recommended, relevant, k))

    return f'Model: {model_func.__name__}, Precision@{k}: {np.mean(precisions):.4f}'

# **Run all Models**

In [121]:
cosine_results = evaluate_model(recommend_cosine)
knn_results = evaluate_model(recommend_knn)
svd_results = evaluate_model(recommend_svd)

In [122]:
print(cosine_results)
print(knn_results)
print(svd_results)

Model: recommend_cosine, Precision@10: 0.0051
Model: recommend_knn, Precision@10: 0.0048
Model: recommend_svd, Precision@10: 0.0046
